In [24]:
import numpy as np
import pandas as pd
from surprise import Dataset, Reader, KNNBasic, AlgoBase
from surprise.model_selection import cross_validate, KFold
from surprise import accuracy

In [25]:
def calculate_mae(predictions):
    abs_errors = [abs(p.r_ui - p.est) for p in predictions if p.r_ui is not None]
    if not abs_errors:
        return 0.0
    return np.mean(abs_errors)

def calculate_rmse(predictions):
    squared_errors = [(p.r_ui - p.est)**2 for p in predictions if p.r_ui is not None]
    if not squared_errors:
        return 0.0
    return np.sqrt(np.mean(squared_errors))

In [26]:
# Naive Algorithm (item average)
class NaiveItemAverage(AlgoBase):
    def __init__(self):
        AlgoBase.__init__(self)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        self.item_means = {}
        
        # Calculate average rating for each item
        for i in trainset.all_items():
            ratings = [r for (_, r) in trainset.ir[i]]
            if ratings:
                self.item_means[i] = np.mean(ratings)
            else:
                self.item_means[i] = trainset.global_mean
        return self

    def estimate(self, u, i):
        # Return item mean if known, else global mean
        return self.item_means.get(i, self.trainset.global_mean)


In [27]:
data = Dataset.load_builtin('ml-100k')

# Defining the 5-Fold Cross Validation
kf = KFold(n_splits=5, random_state=42, shuffle=True)

In [28]:
# Configurations
configurations = [
    {
        'name': 'Naive (Item Average)',
        'algo': NaiveItemAverage(),
        'params': 'None'
    },
    {
        'name': 'KNN Basic (Cosine)',
        'algo': KNNBasic(k=40, min_k=1, sim_options={'name': 'cosine', 'user_based': True}, verbose=False),
        'params': 'k=40, sim=cosine'
    },
    {
        'name': 'KNN Basic (MSD)',
        'algo': KNNBasic(k=40, min_k=1, sim_options={'name': 'msd', 'user_based': True}, verbose=False),
        'params': 'k=40, sim=MSD'
    },
    {
        'name': 'KNN Basic (Pearson)',
        'algo': KNNBasic(k=40, min_k=1, sim_options={'name': 'pearson', 'user_based': True}, verbose=False),
        'params': 'k=40, sim=pearson'
    },
    {
        'name': 'KNN Basic (High Neighbors)',
        'algo': KNNBasic(k=60, min_k=1, sim_options={'name': 'msd', 'user_based': True}, verbose=False),
        'params': 'k=60, sim=MSD'
    },
    {
        'name': 'KNN Basic (Low Neighbors)',
        'algo': KNNBasic(k=10, min_k=1, sim_options={'name': 'msd', 'user_based': True}, verbose=False),
        'params': 'k=10, sim=MSD'
    }
]

In [29]:
results = []
for config in configurations:
    algo_name = config['name']
    algo = config['algo']
    params = config['params']
    
    mae_scores = []
    rmse_scores = []

    # Cross Validation
    for trainset, testset in kf.split(data):
        algo.fit(trainset)
        predictions = algo.test(testset)
        
        mae_scores.append(calculate_mae(predictions))
        rmse_scores.append(calculate_rmse(predictions))
    
    avg_mae = np.mean(mae_scores)
    avg_rmse = np.mean(rmse_scores)
    
    results.append({
        'Algorithm': algo_name,
        'Parameters': params,
        'MAE': avg_mae,
        'RMSE': avg_rmse
    })
    
print(pd.DataFrame(results))

                    Algorithm         Parameters       MAE      RMSE
0        Naive (Item Average)               None  0.817383  1.024777
1          KNN Basic (Cosine)   k=40, sim=cosine  0.804390  1.017077
2             KNN Basic (MSD)      k=40, sim=MSD  0.773483  0.978931
3         KNN Basic (Pearson)  k=40, sim=pearson  0.803166  1.011994
4  KNN Basic (High Neighbors)      k=60, sim=MSD  0.778333  0.983958
5   KNN Basic (Low Neighbors)      k=10, sim=MSD  0.779263  0.988839
